# Load dataset

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
df = pd.read_csv('clean_data.csv')

# Split data

## Original data

In [2]:
#Split the original data into features and target
from sklearn.model_selection import train_test_split


X = df.drop('classification',axis=1)
y = df['classification']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=10, stratify=y)

## Feature selection

### Filter method : correlation-based feature selection

In [3]:
correlation_matrix = df.corr()
correlation = correlation_matrix['classification'].sort_values(ascending=False)
cor_index = list((correlation.index[index]) for index, attr in enumerate(correlation) if attr > 0.5 or attr < -0.5)
X_column_corr = cor_index

In [4]:
filter_data = df[X_column_corr]
filter_data

,classification,htn,dm,sc,al,rc,sg,pcv,hemo
0,1,1,1,-0.395962,1.0,5.869015e-01,1.020,0.628372,1.059271
1,1,0,0,-0.801232,4.0,-1.058283e-15,1.020,-0.108649,-0.452097
2,1,0,1,0.211943,2.0,-1.058283e-15,1.010,-0.968506,-1.078762
3,1,1,0,2.238293,4.0,-9.620758e-01,1.005,-0.845669,-0.488960
4,1,0,0,-0.193327,2.0,-1.280111e-01,1.010,-0.477159,-0.341509
...,...,...,...,...,...,...,...,...,...
395,0,0,0,-1.105185,0.0,2.294455e-01,1.020,0.996882,1.169859
396,0,0,0,-0.395962,0.0,1.778423e+00,1.025,1.856739,1.464760
397,0,0,0,-1.003867,0.0,8.252061e-01,1.020,1.242555,1.206722
398,0,0,0,-0.598597,0.0,1.420967e+00,1.025,1.488229,0.616920


### Wrapper method : Stepwise method

In [5]:
from sklearn.ensemble import RandomForestClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

rf = RandomForestClassifier()

# Initialize the SequentialFeatureSelector
sfs = SFS(rf,
          k_features='best',  # Select the best subset of features
          forward=True,        # Forward selection
          floating=True,
          scoring='accuracy',
          cv=5)

# Fit the SFS on your data
sfs = sfs.fit(X, y)

# Get the selected feature indices
selected_features = list(sfs.k_feature_idx_)
selected_columns = df.columns[list(sfs.k_feature_idx_)+ [len(df.columns) - 1]]

In [6]:
wrapper_data = df[selected_columns]

### Embbed method : Rain forest

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf=RandomForestClassifier()

#fit the model
rf.fit(X, y)

# get feature importance scores
importance = rf.feature_importances_

# create a DataFrame to store feature importance scores
feature_importance = pd.DataFrame({'feature': X.columns, 'importance': importance})

# sort the features by importance score in descending order
feature_importance = pd.Series(importance, index=X.columns)

num_features = list(range(2, len(df)))
accuracy = []

for i in num_features:
    top_idx = feature_importance.sort_values(ascending=False)[:i].index
    X_train_i, X_test_i = X_train[top_idx], X_test[top_idx]

    accuracy.append(accuracy_score(y_test, RandomForestClassifier().fit(X_train_i, y_train).predict(X_test_i)))

optimal_num = num_features[accuracy.index(max(accuracy))]
top_idx = feature_importance.sort_values(ascending=False)[:optimal_num].index

In [8]:
embbed_data = df[top_idx.append(pd.Index(['classification']))]

# Train model

In [9]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

models = [
    ('LR', LogisticRegression()),
    ('SVM', SVC(gamma='auto')),
    ('Naive', GaussianNB()),
    ('KNN', KNeighborsClassifier(n_neighbors=2))
]

datasets = {
    'df': df,
    'filter_data': filter_data,
    'wrapper_data': wrapper_data,
    'embbed_data': embbed_data
}

# Open a file in write mode to save the results
with open('classification_reports.txt', 'w') as f:
    for dataset_name, dataset in datasets.items():
        print(f"Current Dataset: {dataset_name}", file=f)

        X = dataset.drop('classification', axis=1)
        y = dataset['classification']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=50)

        for model_name, model in models:
            model.fit(X_train, y_train)
            print(f"Model: {model_name}", file=f)

            # Training data
            y_train_pred = model.predict(X_train)
            train_report = classification_report(y_train, y_train_pred)
            print("Train Classification Report:", file=f)
            print(train_report, file=f)
            print(train_report)

            # Test data
            y_test_pred = model.predict(X_test)
            test_report = classification_report(y_test, y_test_pred)
            print("Test Classification Report:", file=f)
            print(test_report, file=f)
            print(test_report)

            print("=" * 40, file=f)

        print("=" * 60, file=f)


              precision    recall  f1-score   support

           0       0.98      0.98      0.98       117
           1       0.99      0.99      0.99       163

    accuracy                           0.99       280
   macro avg       0.99      0.99      0.99       280
weighted avg       0.99      0.99      0.99       280

              precision    recall  f1-score   support

           0       0.91      0.97      0.94        33
           1       0.99      0.97      0.98        87

    accuracy                           0.97       120
   macro avg       0.95      0.97      0.96       120
weighted avg       0.97      0.97      0.97       120

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       117
           1       1.00      0.97      0.98       163

    accuracy                           0.98       280
   macro avg       0.98      0.98      0.98       280
weighted avg       0.98      0.98      0.98       280

              preci